### Start spark session

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("GetData")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262"
    )
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.DefaultAWSCredentialsProviderChain"
    )
    .getOrCreate()
)


:: loading settings :: url = jar:file:/Users/fh/Documents/Sem05/big-data-project/venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/fh/.ivy2/cache
The jars for the packages stored in: /Users/fh/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f04872c3-8d61-4533-8081-d37a52a3bf2d;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 81ms :: artifacts dl 2ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| n

### Get all urls from gdelt metadata and store in dataframe

In [2]:
from pyspark.sql.functions import col

df = spark.read.json("s3a://bigdata-mapping-ai-project/gdelt-articles/20240104160100.json")
df = df.select(col('url'))
df.show()

25/12/19 23:56:00 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+--------------------+
|                 url|
+--------------------+
|https://www.scmp....|
|https://radio.wps...|
|https://www.touch...|
|https://www.thest...|
|https://wflafm.ih...|
|https://www.elwak...|
|https://1037theq....|
|https://kc101.ihe...|
|https://z1043.ihe...|
|https://www.cnbc....|
|https://www.thehi...|
|https://wmeq.ihea...|
|https://westfairo...|
|https://www.thesi...|
|https://www.mypla...|
|https://www.revie...|
|https://www.rttne...|
|https://texasredz...|
|https://www.rte.i...|
|https://www.921th...|
+--------------------+
only showing top 20 rows



### Find common crawl indexes for each url

In [3]:
import requests

indexes  = [
  'CC-MAIN-2024-10',
  'CC-MAIN-2024-18',
  'CC-MAIN-2024-22',
  'CC-MAIN-2024-26',
  'CC-MAIN-2024-30',
  'CC-MAIN-2024-33',
  'CC-MAIN-2024-38',
  'CC-MAIN-2024-42',
  'CC-MAIN-2024-46',
  'CC-MAIN-2024-51'
]

In [4]:
headers = {"User-Agent": "Mozilla/5.0 (compatible; CommonCrawlFetcher/1.0)"}

In [5]:
from urllib.parse import quote_plus

def find_url_in_indexes(url):
  encoded_url = quote_plus(url)
  for index in indexes:
    api_url = f"https://index.commoncrawl.org/{index}-index?url={encoded_url}&output=json"
    response = requests.get(api_url, headers=headers)
    if response.status_code == 200 and response.text.strip():
      return index
  return None

In [6]:
count = 0
for url in df.collect():
  result = find_url_in_indexes(url['url'])
  print(result)
  count += 1
  if count > 0:
    break

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

### Fetch warc content

In [ ]:
def get_warc_info(url, indexes):
  encoded_url = quote_plus(url)
  for index in indexes:
    api_url = f"https://index.commoncrawl.org/{index}-index?url={encoded_url}&matchType=exact&output=json"
    try:
      response = requests.get(api_url, headers=headers, timeout=10)
      if response.status_code == 200 and response.text.strip():
        first_line = response.text.strip().splitlines()[0]
        import json
        record = json.loads(first_line)
        return {
            "filename": record["filename"],
            "offset": int(record["offset"]),
            "length": int(record["length"])
        }
    except requests.RequestException:
      continue
  return None


In [ ]:
import gzip
import io

def fetch_warc_record(filename, offset, length):
    url = f"https://data.commoncrawl.org/{filename}"
    headers_range = {
        "Range": f"bytes={offset}-{offset + length - 1}",
        "User-Agent": headers["User-Agent"]
    }

    response = requests.get(url, headers=headers_range, timeout=20)
    response.raise_for_status()

    # Decompress WARC in memory
    with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as gz:
        warc_record = gz.read().decode("utf-8", errors="replace")
    
    return warc_record


In [ ]:
def extract_html_from_warc(warc_record):
    # WARC headers + HTTP headers end with a double newline
    split_marker = "\r\n\r\n"
    parts = warc_record.split(split_marker, 2)
    if len(parts) == 3:
        html = parts[2]
        return html
    return None


### Save in S3

In [ ]:
import boto3

s3 = boto3.client("s3")  # Make sure AWS credentials are configured

def upload_html_to_s3(html_content, bucket_name, key):
    s3.put_object(Bucket=bucket_name, Key=key, Body=html_content, ContentType="text/html")